# Anima: a nano body finetuned on captions reads new wordings and descriptions that name no side

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/main/notebooks/anima_colab_caption_readers.ipynb)

Read in the framed grammar, every earlier reader set the three sliders precisely on wordings whose words it had trained on, and
an untrained reader of the same shape did as well; no byte-level reader read wordings that share no word with training; no
reader left the sliders at zero on descriptions that name no side. Here the nano body Pollux dual, finetuned on captions in which
each attribute comes in many wordings, reads two tiers beside its control (the same captions, every frame taken from another
caption), its start and its untrained body: 32 descriptions whose wordings it never trained on (flaxen hair, a sullen
expression, cornflower eyes) and 32 that name no side (long hair, a composed look, small eyes). The readings were made
beforehand and live in the data repo's `beatrix/cgc_readers.json`.

**Nothing is drawn twice.** A picture depends only on its character, seed and the three amounts, so a description read at
zero everywhere is the plain picture itself (copied), and any picture already drawn is copied too; the plain prompt's pictures
and the group at amount 1 come from
[AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data); each reader's
set is saved there under `sets/cgc2_*` as soon as it is drawn.

**Content screen.** As before: a picture rated questionable or explicit, or read as child, loli or aged down above 0.2, is
left off the sheets and its set is not uploaded; the fourth character's pictures carry "child, loli, aged down" in the
negative prompt.

**The rules (fixed before the run).** New wordings: for each reader, pair and side, the 16 descriptions' pictures each against
their cell's plain picture, carried when at least 12 of 16 move the side's way beyond 3 standard errors; a pair a control
carries is credited to no reader. No side: a pair stays plain when at least 24 of 32 amounts round to zero and the pictures
move, on average, less than a quarter of the group's full move; the finetuned body is credited against its control.

**Cost.** At most 256 pictures at 768 px and 30 steps (many copied); about two hours on the RTX PRO 6000 at most. Results:
`experiments/e059_anima_caption_readers` of
[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima).


## 0 · Confirm the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap: the runner at its release, and the dependencies (restart ONCE the first time)

Clones the runner at the release this notebook was written for, and the diffusion-pipe fork beside it (its Anima model
code draws the pictures), then installs the dependencies. **Run the cell; if the kernel restarts, run it again** (it
then skips the install).

In [ ]:
import os, sys, subprocess
TAG = "v0.11.0"                       # this notebook's release of the runner
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", TAG], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the diffusion-pipe fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · Set up

Checks the GPU, finds the fork and downloads the three model files. Needs a **write** `HF_TOKEN` in Colab Secrets: the
results and the drawn sets are uploaded.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; results -> AbstractPhil/geolip-beatrix-anima, sets -> ...-anima-data
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

## 3 · The readers on both tiers

Downloads the readers file, prints the job's size (which sets come from the data repo, how many distinct pictures it
draws), then time spent and time left with one line per set, and at the end one line per tier and reader with its verdict. A
run already done is skipped (`force=True` reruns the reads; the sets come back from the data repo).


In [ ]:
from huggingface_hub import hf_hub_download
readers = hf_hub_download("AbstractPhil/geolip-beatrix-anima-data", "beatrix/cgc_readers.json", repo_type="dataset")
meta = a.run_group_readers(readers, test="e059")     # 4 readers x (32 new wordings + 32 descriptions that name no side)


## Notes

- **Where things land:** `experiments/e059_anima_caption_readers/` in the experiments repo (README with the question,
  the design, the rules fixed before the run and the result; `meta.json`; `readers.json`; `result.json` with every picture's
  score on every pair; `tagger_probs.npz`; `sheet_unseen.jpg`, `sheet_off.jpg`). The sets:
  `sets/cgc2_<reader>_<tier>-<key>/` in the data repo. Locally: `/content/anima_data/`.
- **After a disconnect:** run steps 1-3 again. Finished sets come back from the data repo; only the unfinished set is
  drawn again.
- **The pictures** are outputs of Anima-Base v1.0 (CircleStone Labs Non-Commercial License: outputs are not derivatives
  of the model; the restriction applies to the model and its derivatives, such as LoRAs).
